# Phase 6: Reviewed outreach adapter

Run this notebook on a private Colab GPU after reviewing the synthetic pilot manifest. It trains one bounded LoRA/QLoRA adapter, saves artifacts to private Drive, and keeps the Phase 1 benchmark held out. It does not expose a model endpoint or send outreach.

In [ ]:
%pip install -q transformers==5.14.1 accelerate==1.14.0
%pip install -q bitsandbytes==0.50.0 peft==0.20.0 safetensors==0.8.0

import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/muzzary/GTM-Agent.git"
REPOSITORY_REF = "codex/phase-6-reviewed-adapter"
REPOSITORY_DIR = Path("/content/GTM-Agent")
if not REPOSITORY_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPOSITORY_REF,
            REPOSITORY_URL,
            str(REPOSITORY_DIR),
        ],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPOSITORY_DIR)],
    check=True,
)
sys.path.insert(0, str(REPOSITORY_DIR))

In [ ]:
import gc
import hashlib
import json
import random
import time
from datetime import UTC, datetime

import torch
from google.colab import drive

from src.evaluation.phase1 import load_manifest, parse_model_output
from src.evaluation.phase5 import run_baseline
from src.evaluation.phase5_io import save_baseline_report
from src.evaluation.phase6 import compare_baseline_reports
from src.evaluation.phase6_io import save_comparison_report
from src.schemas.dataset import DatasetManifest
from src.schemas.inference import (
    GenerationSettings,
    InferenceResponse,
    ModelIdentity,
    RuntimeMetadata,
)
from src.schemas.training import AdapterArtifactMetadata, TrainingConfig
from src.training.dataset import validate_dataset

CONFIG_PATH = REPOSITORY_DIR / "configs/phase6/training.json"
DATASET_PATH = REPOSITORY_DIR / "configs/phase6/pilot.json"
BENCHMARK_PATH = REPOSITORY_DIR / "configs/phase1/benchmark.json"
config_model = TrainingConfig.model_validate(
    json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
)
config = config_model.model_dump()
dataset = DatasetManifest.model_validate(
    json.loads(DATASET_PATH.read_text(encoding="utf-8"))
)
benchmark = load_manifest(BENCHMARK_PATH)
audit = validate_dataset(dataset, benchmark)
assert audit.passed
assert audit.split_counts["train"] > 0
assert audit.split_counts["held_out"] > 0
assert torch.cuda.is_available(), "Phase 6 requires a private Colab GPU runtime."
assert config["base_model_id"] and config["base_model_revision"]
random.seed(config["seed"])
torch.manual_seed(config["seed"])
drive.mount("/content/drive")

In [ ]:
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

APPROVED_MODEL_ID = ""
APPROVED_MODEL_REVISION = ""
assert APPROVED_MODEL_ID == config["base_model_id"], (
    "Set the reviewed base model ID before training."
)
assert APPROVED_MODEL_REVISION == config["base_model_revision"], (
    "Set the reviewed base revision before training."
)
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(
    APPROVED_MODEL_ID,
    revision=APPROVED_MODEL_REVISION,
    trust_remote_code=False,
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    APPROVED_MODEL_ID,
    revision=APPROVED_MODEL_REVISION,
    quantization_config=quantization,
    device_map="auto",
    trust_remote_code=False,
    use_safetensors=True,
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(
        r=config["lora_rank"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=config["target_modules"],
    ),
)
model.print_trainable_parameters()

In [ ]:
def training_text(example):
    return (
        example.prompt
        + "\n\nApproved response:\nSubject: "
        + example.target_subject
        + "\nBody:\n"
        + example.target_body
    )

train_examples = [
    example for example in dataset.examples if example.split.value == "train"
]
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
model.train()
steps = 0
for _epoch in range(config["epochs"]):
    for example in train_examples:
        encoded = tokenizer(
            training_text(example),
            return_tensors="pt",
            truncation=True,
            max_length=config["max_length"],
        )
        encoded = {key: value.to(model.device) for key, value in encoded.items()}
        loss = model(**encoded, labels=encoded["input_ids"]).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        steps += 1
        if steps >= config["max_steps"]:
            break
    if steps >= config["max_steps"]:
        break
assert steps > 0 and steps <= config["max_steps"]
print({"steps": steps, "last_loss": float(loss.detach().cpu())})

In [ ]:
ARTIFACT_ROOT = Path("/content/drive/MyDrive/gtm-agent-phase6")
ADAPTER_DIR = ARTIFACT_ROOT / config["adapter_id"]
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)
adapter_files = sorted(path for path in ADAPTER_DIR.rglob("*") if path.is_file())
adapter_sha256 = hashlib.sha256(
    b"".join(path.read_bytes() for path in adapter_files)
).hexdigest()
metadata = {
    "artifact_version": "1.0",
    "adapter_id": config["adapter_id"],
    "adapter_revision": adapter_sha256,
    "base_model_id": APPROVED_MODEL_ID,
    "base_model_revision": APPROVED_MODEL_REVISION,
    "dataset_id": dataset.dataset_id,
    "dataset_version": dataset.dataset_version,
    "train_examples": len(train_examples),
    "trained_steps": steps,
    "created_at": datetime.now(UTC).isoformat(),
}
(ADAPTER_DIR / "adapter-metadata.json").write_text(
    json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8"
)
print(metadata)

In [ ]:
metadata_model = AdapterArtifactMetadata.model_validate(metadata)
assert metadata_model.base_model_id == APPROVED_MODEL_ID
assert metadata_model.base_model_revision == APPROVED_MODEL_REVISION
EVAL_GENERATION = GenerationSettings(
    max_new_tokens=benchmark.generation.max_new_tokens,
    seed=benchmark.generation.seed,
)
BASE_MODEL_IDENTITY = ModelIdentity(
    model_id=APPROVED_MODEL_ID,
    model_revision=APPROVED_MODEL_REVISION,
)
ADAPTER_MODEL_IDENTITY = ModelIdentity(
    model_id=APPROVED_MODEL_ID,
    model_revision=APPROVED_MODEL_REVISION,
    adapter_id=metadata_model.adapter_id,
    adapter_revision=metadata_model.adapter_revision,
)

def load_evaluation_model(adapter_dir=None):
    eval_tokenizer = AutoTokenizer.from_pretrained(
        APPROVED_MODEL_ID,
        revision=APPROVED_MODEL_REVISION,
        trust_remote_code=False,
        use_fast=True,
    )
    eval_model = AutoModelForCausalLM.from_pretrained(
        APPROVED_MODEL_ID,
        revision=APPROVED_MODEL_REVISION,
        quantization_config=quantization,
        device_map="auto",
        trust_remote_code=False,
        use_safetensors=True,
    )
    if adapter_dir is not None:
        eval_model = PeftModel.from_pretrained(
            eval_model, adapter_dir, is_trainable=False
        )
    return eval_tokenizer, eval_model

def generate_evaluation_response(request, eval_tokenizer, eval_model, identity):
    messages = [
        {"role": "system", "content": "Return strict JSON only."},
        {"role": "user", "content": request.prompt},
    ]
    encoded = eval_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(eval_model.device)
    torch.manual_seed(request.seed)
    torch.cuda.manual_seed_all(request.seed)
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = eval_model.generate(
            **encoded,
            do_sample=False,
            max_new_tokens=request.max_new_tokens,
            pad_token_id=eval_tokenizer.eos_token_id,
        )
    elapsed_ms = (time.perf_counter() - started) * 1000
    generated = output_ids[0, encoded["input_ids"].shape[-1] :]
    parsed = parse_model_output(
        eval_tokenizer.decode(generated, skip_special_tokens=True).strip()
    )
    return InferenceResponse(
        request_id=request.request_id,
        model=identity,
        generation=EVAL_GENERATION,
        output=parsed,
        runtime=RuntimeMetadata(
            python_version="3.12",
            torch_version=torch.__version__,
            transformers_version="5.14.1",
            cuda_version=torch.version.cuda,
            gpu_name=torch.cuda.get_device_name(0),
            gpu_memory_mb=(
                torch.cuda.get_device_properties(0).total_memory // (1024 * 1024)
            ),
            latency_ms=elapsed_ms,
        ),
    )

def evaluate_loaded_model(eval_tokenizer, eval_model, identity):
    eval_model.eval()
    return run_baseline(
        benchmark,
        identity,
        lambda request: generate_evaluation_response(
            request, eval_tokenizer, eval_model, identity
        ),
        max_retries=0,
        max_new_tokens=EVAL_GENERATION.max_new_tokens,
        seed=EVAL_GENERATION.seed,
    )

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
eval_tokenizer, eval_base_model = load_evaluation_model()
base_report = evaluate_loaded_model(
    eval_tokenizer, eval_base_model, BASE_MODEL_IDENTITY
)
del eval_base_model
torch.cuda.empty_cache()
eval_tokenizer, eval_adapter_model = load_evaluation_model(ADAPTER_DIR)
adapter_report = evaluate_loaded_model(
    eval_tokenizer, eval_adapter_model, ADAPTER_MODEL_IDENTITY
)
comparison = compare_baseline_reports(
    base_report,
    adapter_report,
    hashlib.sha256(BENCHMARK_PATH.read_bytes()).hexdigest(),
)
save_baseline_report(base_report, ARTIFACT_ROOT / "base-report.json")
save_baseline_report(adapter_report, ARTIFACT_ROOT / "adapter-report.json")
save_comparison_report(comparison, ARTIFACT_ROOT / "comparison.json")
print(comparison.model_dump(mode="json"))

## Evaluation gate

Do not evaluate on the training examples. Generate adapter outputs only for the unchanged Phase 5 benchmark and compare them with the saved base report using the existing `InferenceResponse` and `BaselineReport` contracts. A report is not accepted unless approved-claim and evidence gates remain clean.